# (3) Perform negative sampling to get more non-SL pairs

In [6]:
import pandas as pd
import pickle as pkl
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pandarallel import pandarallel
from ast import literal_eval

pandarallel.initialize(progress_bar=True)
tqdm.pandas()

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [2]:
sub_embs = pd.read_csv("/work/magroup/kaileyhu/Cilantro-SL/outputs/gf_12L_30M_i2048_SL/generated_df/sub_embs_only.csv")

In [3]:
sub_embs.to_hdf("/work/magroup/kaileyhu/Cilantro-SL/outputs/gf_12L_30M_i2048_SL/generated_df/sub_embs_only.hdf", key="table")

In [7]:
sub_embs.set_index("Unnamed: 0", inplace = True)

In [8]:
sub_embs['gene'] = list(map(lambda x : literal_eval(x)[1], list(sub_embs.index)))
sub_embs['gene'] = sub_embs['gene'].apply(lambda x : x[5:])
valid_ensembl = set(sub_embs['gene'])

In [9]:
SL_df = pd.read_csv("/work/magroup/kaileyhu/Cilantro-SL/data/Human_SL.csv")
nonSL_df = pd.read_csv("/work/magroup/kaileyhu/Cilantro-SL/data/Human_nonSL.csv")

In [10]:
# Pull all pos / neg pairs. If a pair is present in both, delete it

res_pairs = {}
for i in range(len(SL_df)):
    row = SL_df.iloc[i]
    g1 = row["n1.name"]
    g2 = row["n2.name"]
    res_pairs[(g1, g2)] = True

num_overlap = 0
for i in range(len(nonSL_df)):
    row = nonSL_df.iloc[i]
    g1 = row["n1.name"]
    g2 = row["n2.name"]
    if (g1, g2) in res_pairs:
        num_overlap += 1
        del res_pairs[(g1, g2)]
    else:
        res_pairs[(g1, g2)] = False

In [12]:
with open ("/work/magroup/kaileyhu/Cilantro-SL/data/all_pairs_dict.pkl", "wb") as f:
    pkl.dump(res_pairs, f)

pair_list = res_pairs

In [13]:
df = pd.read_csv("/work/magroup/kaileyhu/Cilantro-SL/data/OmicsExpressionProteinCodingGenesTPMLogp1.csv")

In [ ]:
df.set_index("Unnamed: 0", inplace = True)
corr_mat = df.corr()

corr_mat.to_csv("/work/magroup/kaileyhu/Cilantro-SL/data/temp/NSM_EXP.csv")
corr_mat = pd.read_csv("/work/magroup/kaileyhu/Cilantro-SL/data/temp/NSM_EXP.csv")

In [ ]:
corr_mat.set_index("Unnamed: 0", inplace = True)
corr_mat.index = list(map(lambda x : x.split(' ')[0], corr_mat.index))
corr_mat.columns = list(map(lambda x : x.split(' ')[0], corr_mat.columns))
corr_dict = corr_mat.to_dict()

In [ ]:
res = set()
for (k, v) in tqdm(corr_dict.items()):
    for (k2, v2) in v.items():
        if k < k2:
            res.add((k, k2, v2))
        else:
            res.add((k2, k, v2))

sorted_items = sorted(res, key=lambda item: item[2])

In [ ]:
with open ("/work/magroup/kaileyhu/Cilantro-SL/data/temp/nsm_exp_sorted.pkl", "wb") as f:
    pkl.dump(sorted_items, f)

In [ ]:
ensembl_path = "/work/magroup/kaileyhu/Cilantro-SL/data/ensembl_mapping_dict_gc95M.pkl"

def invert_dict(dict_obj):
    return {v: k for k, v in dict_obj.items()}

with open(ensembl_path, "rb") as f:
    id_gene_dict = pkl.load(f)

def query_id(g):
    if g in id_gene_dict:
        if id_gene_dict[g] in valid_ensembl:
            return id_gene_dict[g]
    return " "

In [ ]:
# subset pairlist to only valid genes
pair_list = {(g1, g2) : v for ((g1, g2), v) in pair_list.items() if query_id(g1) != " " and query_id(g2) != " "}

pos_pairs = {key: value for key, value in pair_list.items() if value}

sorted_items_f = list(filter(lambda x :  query_id(x[0]) != " " and  query_id(x[1]) != " ", sorted_items))

In [ ]:
includes_all = []
num_added = 0
sample_param = 5

for i, (g1, g2, corr) in tqdm(enumerate(sorted_items_f)):
    g1_ens, g2_ens = query_id(g1), query_id(g2)
    if g1_ens == " " or g2_ens == " ":
        continue
    if (g1, g2, corr) in includes_all or (g1, g2) in pair_list:
        continue
    if len(includes_all) + num_neg >= sample_param * num_pos:
        break
    includes_all.append((g1, g2, corr))
    num_added += 1
    if (num_added % 10000 == 0):
        print(f"added {num_added} so far")

In [ ]:
for (g1, g2, _) in includes_all:
    pair_list[(g1, g2)] = False

In [ ]:
len(pair_list), len({key: value for key, value in pair_list.items() if value}), len({key: value for key, value in pair_list.items() if not value})

In [ ]:
with open ("/work/magroup/kaileyhu/Cilantro-SL/data/processed/all_pairs_NSM_EXP.pkl", "wb") as f:
    pkl.dump(pair_list, f)